# 02 — Transfer Learning dengan ResNet18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/02-cnn/02_transfer_learning.ipynb)

Melatih CNN dari nol butuh data yang banyak. Jika datanya sedikit, kita bisa memakai model yang sudah dilatih di dataset besar (**ImageNet**: 1,2 juta gambar, 1000 kelas), lalu menyesuaikannya untuk tugas kita. Teknik ini disebut **transfer learning**.

Idenya: layer awal CNN mempelajari fitur umum (tepi, warna, tekstur) yang berguna untuk hampir semua gambar. Kita cukup mengganti layer terakhir (*classifier head*) agar sesuai dengan kelas baru.

Kita bandingkan dua strategi:

| Strategi | Yang dilatih | Kelebihan | Kekurangan |
|---|---|---|---|
| **Freeze backbone** | hanya layer `fc` terakhir | cepat, hemat memori, tidak mudah overfitting | fitur tidak bisa beradaptasi ke domain baru |
| **Fine-tune semua layer** | semua layer | akurasi biasanya lebih tinggi | lebih lambat, perlu learning rate kecil agar fitur pretrained tidak rusak |

**Dataset**: [Oxford Flowers102](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/), 102 jenis bunga. Split train hanya berisi **10 gambar per kelas** (1.020 gambar), kasus klasik di mana transfer learning sangat membantu.

> Notebook ini dirancang untuk **Google Colab dengan GPU**.

## 0. Cek GPU

Jalankan sel ini paling awal. Jika GPU belum aktif, di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas.

Notebook tetap bisa berjalan di CPU, tetapi training akan jauh lebih lambat. Untuk uji cepat di CPU, gunakan `QUICK_RUN` (lihat bagian konfigurasi).

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, training akan berjalan di CPU (lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

# nvidia-smi menampilkan detail GPU & pemakaian memori (hanya ada jika driver NVIDIA terpasang)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Setup dan Konfigurasi

`EPOCHS` berlaku untuk masing-masing strategi, jadi total ada 2 × `EPOCHS` epoch training.

`QUICK_RUN=1` (environment variable) memakai subset sangat kecil dan 1 epoch untuk uji cepat di CPU.

In [ ]:
import os
import time
import copy
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
BATCH_SIZE = 32
EPOCHS = 1 if QUICK_RUN else 10
DATA_DIR = "./data"
NUM_WORKERS = 2

print("device:", device, "| QUICK_RUN:", QUICK_RUN, "| EPOCHS per strategi:", EPOCHS)

## 2. Data dan Transformasi

Model pretrained mengharapkan input dengan format yang sama seperti saat ia dilatih di ImageNet:
- Ukuran **224×224**.
- Dinormalisasi dengan **mean/std ImageNet**.

Transformasi train memakai augmentation (`RandomResizedCrop` memotong area acak lalu mengubahnya ke 224×224, ditambah flip). Transformasi evaluasi hanya *resize* dan *center crop*.

Flowers102 sudah menyediakan split `train`, `val`, dan `test`, jadi kita tidak perlu membaginya sendiri. Unduhan pertama sekitar 330 MB.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = datasets.Flowers102(DATA_DIR, split="train", download=True, transform=train_transform)
val_set = datasets.Flowers102(DATA_DIR, split="val", download=True, transform=eval_transform)
test_set = datasets.Flowers102(DATA_DIR, split="test", download=True, transform=eval_transform)
CLASS_NAMES = train_set.classes
N_CLASSES = len(CLASS_NAMES)

if QUICK_RUN:
    # Subset sangat kecil; diacak dulu agar berisi beberapa kelas berbeda
    g = torch.Generator().manual_seed(0)
    train_set = Subset(train_set, torch.randperm(len(train_set), generator=g)[:64].tolist())
    val_set = Subset(val_set, torch.randperm(len(val_set), generator=g)[:32].tolist())
    test_set = Subset(test_set, torch.randperm(len(test_set), generator=g)[:32].tolist())

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)

print(f"kelas: {N_CLASSES} | train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")

Contoh gambar dari satu batch train (sudah melalui augmentation):

In [ ]:
def denormalize(img):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1).permute(1, 2, 0)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, img, label in zip(axes.flat, images, labels):
    ax.imshow(denormalize(img))
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Mengenal ResNet18

**ResNet** memperkenalkan *residual connection* (jalan pintas `x + f(x)`) yang memungkinkan jaringan sangat dalam tetap mudah dilatih. ResNet18 memiliki 18 layer berbobot dan sekitar 11,7 juta parameter.

Strukturnya: `conv1 → layer1 → layer2 → layer3 → layer4 → avgpool → fc`. Bagian sebelum `fc` disebut **backbone** (pengekstrak fitur), sedangkan `fc` adalah **head** (classifier).

Bobot pretrained ImageNet diunduh otomatis saat pertama kali dipakai (sekitar 45 MB).

In [ ]:
WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1

resnet = models.resnet18(weights=WEIGHTS)
print("Layer tingkat atas:", [nama for nama, _ in resnet.named_children()])
print("Head asli:", resnet.fc)  # 512 fitur -> 1000 kelas ImageNet
print(f"Total parameter: {sum(p.numel() for p in resnet.parameters()):,}")

## 4. Menyiapkan Model untuk Dua Strategi

Fungsi `build_model(strategi)`:
1. Muat ResNet18 dengan bobot pretrained.
2. Jika `"freeze"`: set `requires_grad = False` untuk **semua** parameter, sehingga autograd tidak menghitung gradiennya dan optimizer tidak mengubahnya.
3. Ganti `fc` dengan `nn.Linear(512, 102)` yang baru. Layer baru otomatis punya `requires_grad = True`, jadi hanya layer ini yang dilatih pada mode freeze.

Detail penting untuk mode freeze: **BatchNorm** tetap memperbarui statistik *running mean/var* saat `model.train()` meskipun bobotnya dibekukan. Agar backbone benar-benar tetap sama, kita paksa semua layer BatchNorm tetap dalam mode `eval()` selama training.

In [ ]:
def build_model(strategi):
    model = models.resnet18(weights=WEIGHTS)
    if strategi == "freeze":
        for p in model.parameters():
            p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)  # head baru, selalu dilatih
    return model.to(device)


def freeze_batchnorm(model):
    # Jaga statistik BatchNorm backbone agar tidak berubah (dipakai di mode freeze)
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


def hitung_parameter(model):
    total = sum(p.numel() for p in model.parameters())
    dilatih = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, dilatih


for s in ["freeze", "finetune"]:
    total, dilatih = hitung_parameter(build_model(s))
    print(f"{s:<9}: parameter dilatih {dilatih:>10,} dari {total:,} ({dilatih / total:.1%})")

## 5. Optimizer untuk Masing-masing Strategi

- **Freeze**: optimizer hanya menerima parameter `fc`, dengan learning rate 1e-3.
- **Fine-tune semua**: dipakai **learning rate berbeda per bagian** (*discriminative learning rate*) lewat *parameter groups*:
  - backbone: `1e-4` (kecil, agar fitur pretrained hanya disesuaikan sedikit),
  - head `fc`: `1e-3` (besar, karena dimulai dari bobot acak).

Kita juga memakai `AdamW` (Adam dengan *weight decay* yang benar) sebagai regularisasi.

In [ ]:
def build_optimizer(model, strategi):
    if strategi == "freeze":
        return torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

    backbone_params = [p for nama, p in model.named_parameters() if not nama.startswith("fc.")]
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 1e-3},
    ], weight_decay=1e-4)

## 6. Fungsi Training dan Evaluasi

Sama seperti notebook sebelumnya, dengan satu tambahan: parameter `strategi` agar BatchNorm dibekukan pada mode freeze.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, strategi):
    model.train()
    if strategi == "freeze":
        freeze_batchnorm(model)
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_benar, total_data = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)
    return total_loss / total_data, total_benar / total_data

## 7. Menjalankan Eksperimen

Fungsi `run_experiment` melatih satu strategi, mencatat riwayat, waktu, dan menyimpan salinan bobot dengan **akurasi validasi terbaik** (`copy.deepcopy(model.state_dict())`). Bobot terbaik itulah yang dipakai untuk evaluasi test.

Pada GPU T4, setiap epoch butuh sekitar 10–20 detik.

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def run_experiment(strategi):
    torch.manual_seed(42)  # inisialisasi head yang sama untuk kedua strategi
    model = build_model(strategi)
    optimizer = build_optimizer(model, strategi)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_state = -1.0, None

    mulai = time.time()
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, strategi)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn)
        for k, v in zip(history, [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(v)
        if va_acc > best_acc:
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
        print(f"[{strategi}] epoch {epoch:2d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f}")
    durasi = time.time() - mulai

    model.load_state_dict(best_state)  # kembalikan bobot terbaik
    _, test_acc = evaluate(model, test_loader, loss_fn)
    return {"model": model, "history": history, "durasi": durasi,
            "best_val_acc": best_acc, "test_acc": test_acc,
            "param_dilatih": hitung_parameter(model)[1]}


hasil = {}
for strategi in ["freeze", "finetune"]:
    hasil[strategi] = run_experiment(strategi)
    print()

## 8. Perbandingan Hasil

### 8.1 Tabel ringkasan

In [ ]:
print(f"{'strategi':<10} {'param dilatih':>14} {'waktu (s)':>10} {'best val acc':>13} {'test acc':>9}")
for strategi, h in hasil.items():
    print(f"{strategi:<10} {h['param_dilatih']:>14,} {h['durasi']:>10.1f} "
          f"{h['best_val_acc']:>13.3f} {h['test_acc']:>9.3f}")

### 8.2 Kurva validasi

Garis putus-putus = train, garis utuh = validasi. Perhatikan kecepatan naik akurasi dan jarak train vs validasi pada kedua strategi.

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
warna = {"freeze": "tab:blue", "finetune": "tab:orange"}
for strategi, h in hasil.items():
    for a, key in [(ax[0], "loss"), (ax[1], "acc")]:
        a.plot(epochs_range, h["history"][f"train_{key}"], "--", color=warna[strategi], alpha=0.6)
        a.plot(epochs_range, h["history"][f"val_{key}"], "o-", color=warna[strategi], label=strategi)
for a, judul in [(ax[0], "Loss"), (ax[1], "Akurasi")]:
    a.set_title(judul + " (putus-putus = train, utuh = validasi)")
    a.set_xlabel("epoch"); a.legend(); a.set_xticks(list(epochs_range))
plt.tight_layout()
plt.show()

### 8.3 Interpretasi

Hasil yang biasanya terlihat di Colab (10 epoch per strategi):
- **Freeze backbone** sudah mencapai akurasi yang cukup tinggi (sekitar 75–85%) hanya dengan melatih ~52 ribu parameter, dan setiap epoch lebih cepat karena gradien backbone tidak dihitung.
- **Fine-tune semua layer** biasanya lebih tinggi lagi (sekitar 85–92%) karena fitur backbone ikut beradaptasi ke gambar bunga, dengan biaya waktu dan memori lebih besar.
- Keduanya jauh di atas CNN yang dilatih dari nol dengan hanya 10 gambar per kelas.

Aturan praktis: mulai dari **freeze** sebagai baseline cepat. Jika data cukup banyak atau domainnya jauh berbeda dari ImageNet (misalnya citra medis), lanjutkan ke **fine-tune semua** dengan learning rate kecil.

> Jika `QUICK_RUN` aktif, angka di atas tidak bermakna karena datanya terlalu sedikit.

## 9. Contoh Prediksi Model Terbaik

Kita ambil model dengan akurasi test tertinggi dan menampilkan prediksinya di beberapa gambar test.

In [ ]:
terbaik = max(hasil, key=lambda s: hasil[s]["test_acc"])
model = hasil[terbaik]["model"]
print("Model terbaik:", terbaik)

model.eval()
x_test, y_test = next(iter(test_loader))
with torch.no_grad():
    probs = torch.softmax(model(x_test.to(device)), dim=1).cpu()
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(13, 5.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize(x_test[i]))
    benar = preds[i] == y_test[i]
    ax.set_title(f"pred: {CLASS_NAMES[preds[i]]} ({probs[i, preds[i]]:.0%})\nasli: {CLASS_NAMES[y_test[i]]}",
                 fontsize=7, color="green" if benar else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Ringkasan

- Transfer learning = memakai ulang fitur dari model pretrained, lalu mengganti head untuk kelas baru.
- Input harus mengikuti format pretraining (224×224, normalisasi ImageNet).
- **Freeze**: `requires_grad = False` untuk backbone, optimizer hanya berisi head, dan BatchNorm dijaga di mode eval.
- **Fine-tune semua**: pakai learning rate kecil untuk backbone dan lebih besar untuk head (*parameter groups*).
- Simpan bobot dengan akurasi validasi terbaik, lalu evaluasi di test.

## Latihan

1. **Strategi ketiga: unfreeze bertahap.** Latih dengan mode freeze selama 3 epoch, lalu *unfreeze* hanya `layer4` (set `requires_grad = True` untuk parameternya, buat ulang optimizer dengan parameter group untuk `layer4` dan `fc`) dan lanjutkan 7 epoch. Bandingkan akurasi test dan waktunya dengan dua strategi di atas.
2. **Pentingnya bobot pretrained.** Jalankan strategi "fine-tune semua" dengan `models.resnet18(weights=None)` (bobot acak, tanpa pretraining) selama jumlah epoch yang sama. Seberapa besar selisih akurasinya? Jelaskan mengapa dataset dengan 10 gambar per kelas sangat bergantung pada pretraining.
3. **Dataset sendiri.** Kumpulkan gambar untuk 3–5 kelas pilihanmu (minimal 20 gambar per kelas), susun dalam folder `data_saya/train/<nama_kelas>/` dan `data_saya/val/<nama_kelas>/`, lalu muat dengan `datasets.ImageFolder` (upload ke Colab atau Google Drive). Latih dengan strategi terbaik dan tampilkan 5 prediksi yang salah. Apa yang membuat gambar tersebut sulit?